# CS2309 — Benchmark Tốc độ & Chất lượng: bản gốc vs (fp16 + cache)

**Mục đích:** Cải thiện (fp16 + EditCache) làm SwiftEdit nhanh hơn đáng kể — nhưng cần
kiểm chứng **có ảnh hưởng chất lượng ảnh không**. Notebook đo song song:

1. **Tốc độ** (wall-clock/edit): bản gốc `fp32` vs bản cải thiện `fp16 + cache`, tách cache-miss / cache-hit, speedup.
2. **VRAM** (peak bộ nhớ GPU): fp16 lưu trọng số nửa kích thước ⇒ giảm VRAM (lý do fp32 từng OOM trên T4 mà fp16 thì không).
3. **Chất lượng**: ảnh bản gốc `fp32` là **ground truth**, ảnh bản cải thiện là **ảnh cần kiểm tra**, chấm bằng các độ đo được công nhận rộng rãi:
   - **PSNR** (dB, cao hơn = giống hơn)
   - **SSIM** (Wang 2004 — cấu trúc, 0–1)
   - **LPIPS** (Zhang 2018 — cảm nhận/perceptual, thấp hơn = giống hơn)
   - **MSE** (thấp hơn = giống hơn)

Mỗi ảnh chạy **3 prompt khác nhau**. Số ảnh `N` và mọi tham số đều chỉnh được ở cell cấu hình.

> **Lưu ý:** Cache là **lossless** (tái dùng latent/embedding y hệt) → nếu có khác biệt chất lượng, nó đến từ **fp16**, không phải cache.

### Setup (giống `CS2309_SwiftEdit_phase3.ipynb`)

| Môi trường | Thứ tự |
|---|---|
| **Colab T4** | Cell **1** clone + GPU check → Cell **2/3** pip/weights/HF |
| **macOS** | Cell **1** (nhận repo local) → Cell **2/3** `setup_macos.sh` |

Dataset tự dò theo thứ tự: `data/PIE-Bench-subset20` → `data/PIE-Bench` → `data/PIE-Bench-smoke` → `imgs_demo` (fallback). Trên Colab (clone repo) sẽ dùng `PIE-Bench-smoke`/`imgs_demo` nếu chưa tải PIE-Bench đầy đủ.

In [ ]:
import os

# Giảm phân mảnh VRAM trên CUDA/T4 (đặt TRƯỚC khi import torch). Quan trọng với baseline fp32 nặng VRAM.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import subprocess
import sys
from pathlib import Path

# Tự nhận Colab (hoặc gán tay IN_COLAB = True/False)
try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False


_COLAB_GPU_ERR = (
    "Colab chưa có GPU.\n"
    "• Colab web: Runtime → Change runtime type → T4 GPU\n"
    "• Colab extension (VS Code/Cursor): Select Kernel → Colab → New Colab Server "
    "→ Hardware accelerator: GPU → T4, rồi Restart kernel\n"
    "• Đang nối server CPU: Colab: Remove Server, tạo server GPU mới (không đổi GPU trên cùng server)"
)


def _check_colab_gpu() -> None:
    """Kiểm tra GPU sớm (trước clone/pip) — không cần torch."""
    r = subprocess.run(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    )
    if r.returncode != 0 or not (r.stdout or "").strip():
        raise RuntimeError(_COLAB_GPU_ERR)
    names = [ln.strip() for ln in r.stdout.strip().splitlines() if ln.strip()]
    print("GPU OK (nvidia-smi):", ", ".join(names))


# --- Colab: chỉ clone repo đề tài → lưu trên /content (không mount Drive) ---
REPO_SLUG = "NguyenKz/CS2309.CH201"  # đổi nếu fork: "user/CS2309.CH201"
USE_PRIVATE_REPO = True  # True chỉ khi repo private + đã thêm secret GITHUB_TOKEN trên Colab UI


def _colab_repo_url():
    public_url = f"https://github.com/{REPO_SLUG}.git"
    if not USE_PRIVATE_REPO:
        return public_url
    try:
        from google.colab import userdata

        token = userdata.get("GITHUB_TOKEN")
        return f"https://{token}@github.com/{REPO_SLUG}.git"
    except Exception as e:
        print(
            "Không lấy được GITHUB_TOKEN (secret Colab). "
            "Dùng repo public hoặc: Colab → 🔑 Secrets → thêm GITHUB_TOKEN, rồi chạy lại.\n"
            f"Chi tiết: {e}\n"
            f"Fallback clone public: {public_url}"
        )
        return public_url


REPO_URL = _colab_repo_url() if IN_COLAB else f"https://github.com/{REPO_SLUG}.git"
COLAB_REPO_DIR = Path("/content/CS2309.CH201")

if IN_COLAB:
    _check_colab_gpu()
    if not (COLAB_REPO_DIR / "SwiftEdit" / "infer.py").exists():
        print(f"Cloning github.com/{REPO_SLUG} → {COLAB_REPO_DIR} ...")
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(COLAB_REPO_DIR)],
            check=True,
        )
    PROJECT_ROOT = COLAB_REPO_DIR
    os.environ.setdefault("HF_HOME", "/content/huggingface")
    os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "900")
else:
    PROJECT_ROOT = Path.cwd()
    if PROJECT_ROOT.name == "notebooks":
        PROJECT_ROOT = PROJECT_ROOT.parent
    elif not ((PROJECT_ROOT / "SwiftEdit" / "infer.py").exists()):
        for p in [Path.cwd(), *Path.cwd().parents]:
            if (p / "SwiftEdit" / "infer.py").exists():
                PROJECT_ROOT = p
                break

SWIFTEDIT_DIR = PROJECT_ROOT / "SwiftEdit"
WEIGHTS_DIR = SWIFTEDIT_DIR / "swiftedit_weights"
OUTPUT_DIR = PROJECT_ROOT / "results" / "notebook"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

os.chdir(SWIFTEDIT_DIR)
if str(SWIFTEDIT_DIR) not in sys.path:
    sys.path.insert(0, str(SWIFTEDIT_DIR))

print("IN_COLAB:", IN_COLAB)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("SWIFTEDIT_DIR:", SWIFTEDIT_DIR)
print("infer.py:", (SWIFTEDIT_DIR / "infer.py").is_file())
print("HF_HOME:", os.environ.get("HF_HOME", "(default ~/.cache)"))
print("Weights OK:", (WEIGHTS_DIR / "inverse_ckpt-120k").is_dir())
if IN_COLAB and not (SWIFTEDIT_DIR / "infer.py").is_file():
    raise FileNotFoundError(
        "Clone xong nhưng thiếu SwiftEdit/ — đảm bảo đã push SwiftEdit lên GitHub (không chỉ notebook)."
    )

### ② Setup — pip, weights, HF

Chạy sau cell 1 (đã clone + `PROJECT_ROOT`). Cần thêm `torchmetrics` cho SSIM/LPIPS (đã có trong requirements; cell dưới tự cài nếu thiếu).

In [ ]:
env = os.environ.copy()
env["REPO_SLUG"] = REPO_SLUG
env["COLAB_REPO_DIR"] = str(COLAB_REPO_DIR)
if IN_COLAB:
    env.setdefault("HF_HOME", "/content/huggingface")
    env.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "900")

if IN_COLAB:
    if not (PROJECT_ROOT / "SwiftEdit" / "infer.py").is_file():
        raise FileNotFoundError(
            f"Chưa có {PROJECT_ROOT}/SwiftEdit — chạy cell 1 (clone) trước."
        )
    setup_sh = PROJECT_ROOT / "scripts" / "setup_colab.sh"
    print("Chạy:", setup_sh)
    subprocess.run(["bash", str(setup_sh)], check=True, env=env, cwd=PROJECT_ROOT)
else:
    setup_sh = PROJECT_ROOT / "scripts" / "setup_macos.sh"
    print("Chạy:", setup_sh)
    subprocess.run(["bash", str(setup_sh)], check=True, env=env, cwd=PROJECT_ROOT)

# Đảm bảo có torchmetrics (SSIM/LPIPS) + pyarrow/pandas (đọc parquet PIE-Bench từ HuggingFace).
_need = []
for _pkg, _imp in [("torchmetrics", "torchmetrics"), ("pyarrow", "pyarrow"), ("pandas", "pandas")]:
    try:
        __import__(_imp)
    except ImportError:
        _need.append(_pkg)
if _need:
    print("Cài:", *_need)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_need], check=True)

sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
from load_setup_env import load_setup_env

PATHS = load_setup_env(PROJECT_ROOT)
for k, v in PATHS.items():
    print(f"{k}: {v}")

## ③ Cấu hình benchmark

Chỉnh `N_IMAGES`, `PROMPTS_PER_IMAGE`, `EDIT_TEMPLATES`, `CONFIGS` tùy ý. Mặc định so
**bản gốc fp32** (ground truth) vs **bản cải thiện fp16 + cache**.

**Số ảnh (`N_IMAGES`)**: PIE-Bench tối đa **700** mẫu. Nếu local chưa đủ và `AUTO_BUILD_FROM_HF=True`,
notebook **tự tải parquet PIE-Bench từ HuggingFace (~68MB, nhanh)** rồi trích đúng `min(N_IMAGES, 700)` ảnh
(không cần commit cả bộ ảnh vào git). Đặt `N_IMAGES` lớn hơn số mẫu ⇒ tự lấy **tối đa có thể**.

In [ ]:
import json

# ===== CẤU HÌNH (chỉnh tự do) =====
N_IMAGES = 5             # số ảnh benchmark (Colab nhiều hơn được; smoke chỉ có 2 ảnh)
PROMPTS_PER_IMAGE = 3    # số prompt khác nhau mỗi ảnh
WARMUP_EDITS = 2         # số edit khởi động mỗi config (nuốt compile MPS) — KHÔNG tính giờ

# Hai cấu hình so sánh: bản GỐC (ground truth) vs bản CẢI THIỆN
CONFIGS = {
    "baseline_fp32":       dict(dtype="fp32", channels_last=False, use_cache=False),
    "improved_fp16_cache": dict(dtype="fp16", channels_last=True,  use_cache=True),
}
REFERENCE = "baseline_fp32"   # ảnh do bản gốc tạo ra = ground truth để chấm chất lượng

# Mỗi ảnh sinh PROMPTS_PER_IMAGE edit prompt từ template:
#   {src}  = original_prompt (mô tả ảnh gốc)
#   {edit} = editing_prompt của dataset
EDIT_TEMPLATES = [
    "{edit}",
    "{src} at night, dark lighting",
    "{src} in winter, covered in snow",
]

# PIE-Bench tối đa 700 mẫu (10 nhóm). Nếu N lớn hơn -> tự cắt còn tối đa có thể lấy.
PIEBENCH_MAX = 700
# Tự tải parquet PIE-Bench từ HuggingFace (~68MB, nhanh) rồi trích đúng N ảnh, khi dataset
# local chưa đủ. Tránh phải commit cả bộ ảnh vào git (pull repo chậm).
AUTO_BUILD_FROM_HF = True

BENCH_IMG_DIR = OUTPUT_DIR / "qsbench_imgs"
BENCH_IMG_DIR.mkdir(parents=True, exist_ok=True)

# Thư mục dataset tự dựng theo số ảnh cần (ưu tiên dùng lại nếu đã đủ).
AUTO_DATASET_DIR = PROJECT_ROOT / "data" / f"PIE-Bench-auto{min(N_IMAGES, PIEBENCH_MAX)}"

DATASET_CANDIDATES = [
    AUTO_DATASET_DIR,
    PROJECT_ROOT / "data" / "PIE-Bench",
    PROJECT_ROOT / "data" / "PIE-Bench-subset20",
    PROJECT_ROOT / "data" / "PIE-Bench-smoke",
]


def _count_samples(root):
    mf = root / "mapping_file.json"
    if not mf.is_file():
        return 0
    try:
        return len(json.loads(mf.read_text()))
    except Exception:  # noqa: BLE001
        return 0


def ensure_dataset(n_images):
    """Đảm bảo có dataset >= n_images (cap 700). Nếu local chưa đủ -> dựng từ HuggingFace."""
    want = min(n_images, PIEBENCH_MAX)
    best = max((_count_samples(r) for r in DATASET_CANDIDATES), default=0)
    if best >= want:
        return
    if not AUTO_BUILD_FROM_HF:
        print(f"[data] Local chỉ có {best} ảnh (< {want}); AUTO_BUILD_FROM_HF=False -> dùng tối đa có sẵn.")
        return
    print(f"[data] Local có {best} ảnh (< {want}). Dựng PIE-Bench {want} mẫu từ HuggingFace ...")
    sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
    try:
        from create_piebench_subset import build_subset

        build_subset(
            out_dir=AUTO_DATASET_DIR,
            max_samples=want,
            per_category=want,  # lớn -> đổ đầy từng nhóm cho tới khi đủ want (tối đa 700)
            cache_dir=PROJECT_ROOT / "data" / ".hf_pie_bench_pp",
        )
    except Exception as e:  # noqa: BLE001
        print(f"[data] Không dựng được từ HF ({e}). Sẽ dùng tối đa dataset local đang có.")


def _demo_jobs():
    """Fallback khi không có dataset PIE-Bench (vd Colab clone chưa tải data)."""
    demo = PROJECT_ROOT / "SwiftEdit" / "assets" / "imgs_demo"
    base = [
        ("dog", demo / "02.jpg", "dog", "dog with mouth opened"),
        ("woman", demo / "woman_face.jpg", "woman", "Taylor Swift"),
    ]
    return [(i, str(p), s, e) for i, p, s, e in base if p.is_file()]


def load_jobs(n_images, prompts_per_image, templates):
    pairs, used = [], "imgs_demo (fallback)"
    for root in DATASET_CANDIDATES:
        mf = root / "mapping_file.json"
        if not mf.is_file():
            continue
        mapping = json.loads(mf.read_text())
        ann = root / "annotation_images"
        for key, v in mapping.items():
            ip = ann / v["image_path"]
            if not ip.is_file():
                continue
            src = v.get("original_prompt", "") or "the image"
            edit = v.get("editing_prompt", src)
            pairs.append((key, str(ip), src, edit))
            if len(pairs) >= n_images:
                break
        if pairs:
            used = root.name
            break
    if not pairs:
        pairs = _demo_jobs()
    pairs = pairs[:n_images]
    jobs = []
    for key, ip, src, edit in pairs:
        edits = [t.format(src=src, edit=edit) for t in templates[:prompts_per_image]]
        jobs.append({"id": key, "img": ip, "src_p": src, "edits": edits})
    return jobs, used


ensure_dataset(N_IMAGES)
JOBS, DATASET_USED = load_jobs(N_IMAGES, PROMPTS_PER_IMAGE, EDIT_TEMPLATES)
n_edit = sum(len(j["edits"]) for j in JOBS)
if len(JOBS) < N_IMAGES:
    print(f"(!) Yêu cầu {N_IMAGES} ảnh nhưng chỉ có {len(JOBS)} -> dùng tối đa {len(JOBS)}.")
print(f"Dataset: {DATASET_USED} | {len(JOBS)} ảnh × {PROMPTS_PER_IMAGE} prompt = {n_edit} edit/config")
for j in JOBS:
    print(f"  {j['id']}: src={j['src_p']!r}")
    for k, e in enumerate(j["edits"]):
        print(f"      [{k}] {e}")

## ④ Chạy benchmark (đo tốc độ + lưu ảnh)

**Chạy tuần tự (sequential)** — không song song hóa:

1. Config **baseline_fp32** chạy hết toàn bộ ảnh × prompt → giải phóng model.
2. Config **improved_fp16_cache** chạy hết tương tự.
3. Trong mỗi config: **1 edit/lần** (ảnh 1 prompt 0 → prompt 1 → … → ảnh 2 …).

Mỗi config nạp model riêng (tránh dùng chung VRAM), warmup vài edit (không tính giờ),
rồi đo wall-clock từng edit. Với config có cache, các prompt **cùng ảnh** chạy nối tiếp:
prompt đầu = **cache-miss** (nạp latent/embed), các prompt sau = **cache-hit**.

In [ ]:
import gc
import os
import time

import torch
from torchvision.utils import save_image

from infer import EditCache, edit_image, get_device
from models import AuxiliaryModel, InverseModel, IPSBV2Model

# ---- Tuần tự: 1 edit/lần, không batch song song (ổn định VRAM + đo thời gian chính xác) ----
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
torch.set_num_threads(1)

device = get_device()
DEVICE_STR = str(device)
print("device:", DEVICE_STR, "| torch:", torch.__version__)
print("Chế độ: TUẦN TỰ — 1 config xong mới config tiếp; 1 edit/lần.")

# Đường dẫn tuyệt đối (bền với cwd) — WEIGHTS_DIR định nghĩa ở cell 1.
WEIGHTS = WEIGHTS_DIR


def _sync():
    if DEVICE_STR.startswith("cuda") and torch.cuda.is_available():
        torch.cuda.synchronize()
    elif DEVICE_STR.startswith("mps") and torch.backends.mps.is_available():
        torch.mps.synchronize()


def _empty_cache():
    """Dọn cache nhẹ (không gc) — gọi sau mỗi edit để tránh tích lũy/OOM trên T4."""
    if DEVICE_STR.startswith("cuda") and torch.cuda.is_available():
        torch.cuda.empty_cache()
    elif DEVICE_STR.startswith("mps") and torch.backends.mps.is_available():
        torch.mps.empty_cache()


def _free():
    gc.collect()
    _empty_cache()


def _reset_peak_vram():
    """Đặt lại bộ đếm peak VRAM trước khi đo 1 config."""
    if DEVICE_STR.startswith("cuda") and torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()


def _peak_vram_mb():
    """Peak VRAM (MB) kể từ lần reset gần nhất. CUDA: max_memory_allocated;
    MPS: driver_allocated_memory (không có peak API nên lấy mức hiện tại)."""
    if DEVICE_STR.startswith("cuda") and torch.cuda.is_available():
        return torch.cuda.max_memory_allocated() / 1e6
    if DEVICE_STR.startswith("mps") and torch.backends.mps.is_available():
        try:
            return torch.mps.driver_allocated_memory() / 1e6
        except Exception:  # noqa: BLE001
            return float("nan")
    return float("nan")


def build_models(cfg):
    inv = InverseModel(
        str(WEIGHTS / "inverse_ckpt-120k"), device=device,
        dtype=cfg["dtype"], channels_last=cfg["channels_last"],
    )
    aux = AuxiliaryModel(device=device, dtype=cfg["dtype"])
    ip = IPSBV2Model(
        str(WEIGHTS / "sbv2_0.5"),
        str(WEIGHTS / "ip_adapter_ckpt-90k/ip_adapter.bin"),
        aux, device=device, with_ip_mask_controller=True,
        dtype=cfg["dtype"], channels_last=cfg["channels_last"],
    )
    return inv, aux, ip


VRAM = {}  # config -> peak VRAM (MB)


def run_config(name, cfg):
    print(f"\n===== {name} ({cfg}) =====")
    _free()
    _reset_peak_vram()  # đo peak VRAM từ lúc bắt đầu nạp model config này
    inv, aux, ip = build_models(cfg)
    vram_model = _peak_vram_mb()  # VRAM sau khi nạp model (trước inference)
    cache = EditCache() if cfg["use_cache"] else None
    out_dir = BENCH_IMG_DIR / name
    out_dir.mkdir(parents=True, exist_ok=True)

    # Warmup (không tính giờ, không dùng cache để không lẫn vào kết quả)
    w = JOBS[0]
    for i in range(WARMUP_EDITS):
        edit_image(w["img"], w["src_p"], w["edits"][i % len(w["edits"])],
                   inv, aux, ip, cache=None)
    _free()

    rows = []
    for j in JOBS:
        for k, e in enumerate(j["edits"]):
            _sync()
            t0 = time.perf_counter()
            res = edit_image(j["img"], j["src_p"], e, inv, aux, ip, cache=cache)
            _sync()
            dt = time.perf_counter() - t0
            p = out_dir / f"{j['id']}_{k}.png"
            save_image(res[-1], str(p))  # chỉ ảnh edited (bỏ ảnh source-recon)
            state = "hit" if (cfg["use_cache"] and k > 0) else ("miss" if cfg["use_cache"] else "off")
            rows.append(dict(config=name, job_id=j["id"], prompt_idx=k, edit_p=e,
                             runtime_s=round(dt, 4), cache=state, out_path=str(p)))
            print(f"  {j['id']} [{k}] {dt:6.2f}s ({state})  {e[:42]}")
            del res
            _empty_cache()  # tránh tích lũy VRAM giữa các edit (OOM trên T4 với fp32)

    vram_peak = _peak_vram_mb()  # peak VRAM toàn bộ (model + inference)
    VRAM[name] = dict(model_mb=vram_model, peak_mb=vram_peak)
    print(f"  VRAM[{name}] model={vram_model:.0f}MB  peak={vram_peak:.0f}MB")
    del inv, aux, ip, cache
    _free()
    return rows


ALL_ROWS = []
for _name, _cfg in CONFIGS.items():
    ALL_ROWS += run_config(_name, _cfg)  # tuần tự: fp32 xong hết mới fp16+cache
print(f"\nXong {len(ALL_ROWS)} lần đo ({len(CONFIGS)} config, chạy tuần tự).")
print("Peak VRAM:", {k: f"{v['peak_mb']:.0f}MB" for k, v in VRAM.items()})

## ⑤ Chấm chất lượng (ground truth = fp32)

Với từng cặp (ảnh, prompt): ảnh bản gốc `fp32` là **target/ground truth**, ảnh bản cải
thiện là **preds**. Tính PSNR, SSIM, LPIPS, MSE.

> LPIPS tải trọng số AlexNet lần đầu (cần internet). Nếu offline/lỗi tải, notebook tự bỏ LPIPS và vẫn báo PSNR/SSIM/MSE.

In [ ]:
import collections

import numpy as np
from PIL import Image


class QualityMetrics:
    """So 2 ảnh (cùng 512x512): PSNR/SSIM/LPIPS/MSE. preds=test, target=ground truth."""

    def __init__(self, device):
        from torchmetrics.image import (
            PeakSignalNoiseRatio,
            StructuralSimilarityIndexMeasure,
        )
        from torchmetrics.regression import MeanSquaredError

        self.device = device
        self.psnr = PeakSignalNoiseRatio(data_range=1.0).to(device)
        self.ssim = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)
        self.mse = MeanSquaredError().to(device)
        self.lpips = None
        try:
            from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity

            self.lpips = LearnedPerceptualImagePatchSimilarity(
                net_type="alex", normalize=True
            ).to(device)
        except Exception as e:  # noqa: BLE001
            print("LPIPS không khả dụng (bỏ qua, vẫn có PSNR/SSIM/MSE):", e)

    def _load(self, p):
        a = np.asarray(Image.open(p).convert("RGB").resize((512, 512)), np.float32) / 255.0
        # .contiguous() vì permute tạo tensor non-contiguous -> MSE/.view() sẽ lỗi.
        return torch.tensor(a).permute(2, 0, 1).unsqueeze(0).contiguous().to(self.device)

    def compare(self, ref_png, test_png):
        r = self._load(ref_png)
        t = self._load(test_png)
        out = {
            "psnr": float(self.psnr(t, r)),
            "ssim": float(self.ssim(t, r)),
            "mse": float(self.mse(t, r)),
        }
        if self.lpips is not None:
            try:
                out["lpips"] = float(self.lpips(t.clamp(0, 1), r.clamp(0, 1)))
            except Exception:  # noqa: BLE001
                out["lpips"] = float("nan")
        return out


qm = QualityMetrics(device)

# (job_id, prompt_idx) -> {config: out_path}
paths = collections.defaultdict(dict)
for _row in ALL_ROWS:
    paths[(_row["job_id"], _row["prompt_idx"])][_row["config"]] = _row["out_path"]

OTHER = [c for c in CONFIGS if c != REFERENCE]
quality_rows = []
for (job_id, k), pc in sorted(paths.items()):
    if REFERENCE not in pc:
        continue
    for c in OTHER:
        if c not in pc:
            continue
        m = qm.compare(pc[REFERENCE], pc[c])  # ref=ground truth, test=cải thiện
        quality_rows.append(dict(job_id=job_id, prompt_idx=k, config=c, **m))
        lp = m.get("lpips", float("nan"))
        print(f"{job_id}[{k}] {c}: PSNR={m['psnr']:6.2f}  SSIM={m['ssim']:.4f}  "
              f"LPIPS={lp:.4f}  MSE={m['mse']:.5f}")
print(f"\nĐã chấm {len(quality_rows)} cặp ảnh.")

## ⑥ Tổng hợp + xuất báo cáo

Gom tốc độ + chất lượng, in bảng, lưu `report.md` + CSV vào
`experimental_data/quality_speed_bench_<ngày>/`.

In [ ]:
import platform
from datetime import date

import pandas as pd

tdf = pd.DataFrame(ALL_ROWS)
qdf = pd.DataFrame(quality_rows)
has_lpips = "lpips" in qdf.columns and qdf["lpips"].notna().any()


def fmt(x, p=2):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return "—"
    return f"{x:.{p}f}"


# ---- Tốc độ ----
base_mean = tdf[tdf.config == REFERENCE]["runtime_s"].mean()
speed = {}
for c in CONFIGS:
    sub = tdf[tdf.config == c]
    rec = dict(mean=sub.runtime_s.mean(), median=sub.runtime_s.median())
    rec["cache_miss_mean"] = sub[sub.cache == "miss"].runtime_s.mean() if (sub.cache == "miss").any() else float("nan")
    rec["cache_hit_mean"] = sub[sub.cache == "hit"].runtime_s.mean() if (sub.cache == "hit").any() else float("nan")
    rec["speedup_overall"] = base_mean / rec["mean"]
    rec["speedup_steady"] = base_mean / rec["cache_hit_mean"] if not np.isnan(rec["cache_hit_mean"]) else float("nan")
    speed[c] = rec

print("=== TỐC ĐỘ (s/edit) ===")
for c, r in speed.items():
    extra = f" | cache_hit={fmt(r['cache_hit_mean'])}s (steady {fmt(r['speedup_steady'])}x)" if not np.isnan(r["cache_hit_mean"]) else ""
    print(f"{c}: mean={r['mean']:.2f} median={r['median']:.2f} speedup={r['speedup_overall']:.2f}x{extra}")

# ---- Chất lượng ----
qsummary = {}
for c in OTHER:
    s = qdf[qdf.config == c]
    qsummary[c] = dict(
        psnr_mean=s.psnr.mean(), psnr_min=s.psnr.min(),
        ssim_mean=s.ssim.mean(), ssim_min=s.ssim.min(),
        mse_mean=s.mse.mean(),
        lpips_mean=(s.lpips.mean() if has_lpips else float("nan")),
        lpips_max=(s.lpips.max() if has_lpips else float("nan")),
        n=len(s),
    )
print("\n=== CHẤT LƯỢNG vs ground truth (fp32) ===")
for c, r in qsummary.items():
    print(f"{c}: PSNR {r['psnr_mean']:.2f}dB (min {r['psnr_min']:.2f}) | SSIM {r['ssim_mean']:.4f} "
          f"| LPIPS {fmt(r['lpips_mean'], 4)} | MSE {r['mse_mean']:.5f}")

# ---- VRAM ----
base_peak = VRAM.get(REFERENCE, {}).get("peak_mb", float("nan"))
print("\n=== VRAM (peak, MB) ===")
for c in CONFIGS:
    v = VRAM.get(c, {})
    pk = v.get("peak_mb", float("nan"))
    saved = (1 - pk / base_peak) * 100 if base_peak and not np.isnan(pk) and not np.isnan(base_peak) else float("nan")
    print(f"{c}: model={fmt(v.get('model_mb'))}MB  peak={fmt(pk)}MB  giảm so {REFERENCE}={fmt(saved, 1)}%")

# ---- Xuất file ----
OUT_EXP = PROJECT_ROOT / "experimental_data" / f"quality_speed_bench_{date.today().isoformat()}"
(OUT_EXP / "images").mkdir(parents=True, exist_ok=True)
tdf.to_csv(OUT_EXP / "timing_raw.csv", index=False)
qdf.to_csv(OUT_EXP / "quality_raw.csv", index=False)

L = [
    "# Benchmark Tốc độ & Chất lượng — bản gốc (fp32) vs cải thiện (fp16 + cache)",
    "",
    f"- **Ngày:** {date.today().isoformat()}",
    f"- **Thiết bị:** `{DEVICE_STR}` | torch {torch.__version__} | {platform.platform()}",
    f"- **Dataset:** {DATASET_USED}",
    f"- **Quy mô:** {len(JOBS)} ảnh × {PROMPTS_PER_IMAGE} prompt = {len(JOBS) * PROMPTS_PER_IMAGE} edit/config "
    f"(warmup {WARMUP_EDITS} edit/config, không tính giờ)",
    f"- **Ground truth:** `{REFERENCE}` (ảnh do bản gốc tạo ra)",
    "",
    "## 1. Tốc độ (wall-clock / edit)",
    "",
    "| Config | Mean (s) | Median (s) | Cache-miss (s) | Cache-hit (s) | Speedup (overall) | Speedup (cache-hit) |",
    "|--------|---------:|-----------:|---------------:|--------------:|------------------:|--------------------:|",
]
for c in CONFIGS:
    r = speed[c]
    L.append(f"| {c} | {fmt(r['mean'])} | {fmt(r['median'])} | {fmt(r['cache_miss_mean'])} | "
             f"{fmt(r['cache_hit_mean'])} | {fmt(r['speedup_overall'])}× | {fmt(r['speedup_steady'])}× |")

# ---- Bảng VRAM ----
base_peak = VRAM.get(REFERENCE, {}).get("peak_mb", float("nan"))
L += ["", "## 2. VRAM (bộ nhớ GPU)", "",
      "| Config | Model (MB) | Peak (MB) | Giảm peak vs " + REFERENCE + " |",
      "|--------|-----------:|----------:|----:|"]
for c in CONFIGS:
    v = VRAM.get(c, {})
    pk = v.get("peak_mb", float("nan"))
    saved = (1 - pk / base_peak) * 100 if base_peak and not np.isnan(pk) and not np.isnan(base_peak) else float("nan")
    L.append(f"| {c} | {fmt(v.get('model_mb'), 0)} | {fmt(pk, 0)} | {fmt(saved, 1)}% |")
L += ["", "> **Model** = VRAM sau khi nạp trọng số (trước inference); **Peak** = đỉnh VRAM cả "
      "quá trình (model + inference). fp16 lưu trọng số nửa kích thước nên giảm cả hai. "
      "Trên MPS dùng `driver_allocated_memory` (xấp xỉ); trên CUDA dùng `max_memory_allocated` (chính xác)."]

L += ["", "## 3. Chất lượng so với ground truth (fp32)", "",
      "Ảnh bản cải thiện được chấm so với ảnh bản gốc (cùng ảnh + cùng prompt).", ""]
head = "| Config | PSNR↑ mean | PSNR min | SSIM↑ mean | SSIM min | "
sep = "|--------|---:|---:|---:|---:|"
if has_lpips:
    head += "LPIPS↓ mean | LPIPS max | "
    sep += "---:|---:|"
head += "MSE↓ mean | N |"
sep += "---:|---:|"
L += [head, sep]
for c in OTHER:
    r = qsummary[c]
    row = f"| {c} | {fmt(r['psnr_mean'])} | {fmt(r['psnr_min'])} | {fmt(r['ssim_mean'], 4)} | {fmt(r['ssim_min'], 4)} | "
    if has_lpips:
        row += f"{fmt(r['lpips_mean'], 4)} | {fmt(r['lpips_max'], 4)} | "
    row += f"{fmt(r['mse_mean'], 5)} | {r['n']} |"
    L.append(row)

_imp = OTHER[0]
_imp_peak = VRAM.get(_imp, {}).get("peak_mb", float("nan"))
_vram_saved = (1 - _imp_peak / base_peak) * 100 if base_peak and not np.isnan(_imp_peak) and not np.isnan(base_peak) else float("nan")
L += [
    "", "## 4. Diễn giải", "",
    "- **PSNR** (dB, ↑): > 30 dB ⇒ khác biệt rất nhỏ, mắt thường khó nhận ra; > 40 dB ⇒ gần như trùng.",
    "- **SSIM** (0–1, ↑): > 0.95 ⇒ giữ gần như nguyên cấu trúc.",
    "- **LPIPS** (↓): < 0.1 ⇒ rất giống về cảm nhận (perceptual).",
    "- **MSE** (↓): càng nhỏ càng giống.",
    "- **VRAM** (↓): fp16 lưu trọng số nửa kích thước ⇒ giảm bộ nhớ, chạy được trên GPU yếu / batch lớn hơn.",
    "",
    f"**Kết luận sơ bộ:** bản cải thiện `{_imp}` nhanh hơn **{fmt(speed[_imp]['speedup_overall'])}×** "
    f"(overall) / **{fmt(speed[_imp]['speedup_steady'])}×** (khi cache-hit), "
    f"**giảm ~{fmt(_vram_saved, 1)}% VRAM** (peak {fmt(_imp_peak, 0)}MB vs {fmt(base_peak, 0)}MB), "
    f"trong khi so với ground truth đạt PSNR **{fmt(qsummary[_imp]['psnr_mean'])} dB**, "
    f"SSIM **{fmt(qsummary[_imp]['ssim_mean'], 4)}**"
    + (f", LPIPS **{fmt(qsummary[_imp]['lpips_mean'], 4)}**" if has_lpips else "")
    + ". ⇒ tốc độ tăng mạnh + tiết kiệm VRAM mà chất lượng gần như không đổi.",
    "",
    "> Cache là **lossless** (tái dùng latent/embedding y hệt) nên khác biệt chất lượng (nếu có) đến từ **fp16**, không phải cache. "
    "Wall-clock có thể nhiễu do thermal throttling trên Mac; per-image median ổn định hơn.",
    "",
    "## Tài liệu metric",
    "- **PSNR/MSE:** chuẩn image fidelity.",
    "- **SSIM:** Wang et al. 2004, *Image Quality Assessment: From Error Visibility to Structural Similarity*, IEEE TIP.",
    "- **LPIPS:** Zhang et al. 2018, *The Unreasonable Effectiveness of Deep Features as a Perceptual Metric*, CVPR.",
    "",
    "## Tái lập",
    "```bash",
    "# Mac hoặc Colab: mở notebook và chạy tuần tự các cell",
    "jupyter lab notebooks/CS2309_SwiftEdit_quality_speed_bench.ipynb",
    "```",
    "Chỉnh `N_IMAGES`, `PROMPTS_PER_IMAGE`, `EDIT_TEMPLATES`, `CONFIGS` ở cell cấu hình (③).",
    "",
    "## File trong thư mục",
    "| File | Nội dung |",
    "|------|----------|",
    "| `report.md` | Báo cáo này |",
    "| `timing_raw.csv` | Thời gian từng edit (config/job/prompt/cache) |",
    "| `quality_raw.csv` | PSNR/SSIM/LPIPS/MSE từng cặp ảnh |",
    "| `images/comparison_grid.png` | Lưới so sánh Input / fp32 (GT) / fp16+cache |",
]

(OUT_EXP / "report.md").write_text("\n".join(L) + "\n", encoding="utf-8")
print("\nReport ->", OUT_EXP / "report.md")
print("CSV ->", OUT_EXP / "timing_raw.csv", "+", OUT_EXP / "quality_raw.csv")
display(tdf)
display(qdf)

## ⑦ So sánh trực quan

Lưới: **Input** | **fp32 (ground truth)** | **fp16 + cache** (kèm PSNR/SSIM/LPIPS từng ảnh).

In [ ]:
import matplotlib.pyplot as plt

_imp = OTHER[0]
n_show = min(4, len(JOBS))
fig, axes = plt.subplots(n_show, 3, figsize=(11, 3.7 * n_show))
if n_show == 1:
    axes = axes[None, :]
for r, j in enumerate(JOBS[:n_show]):
    pc = paths[(j["id"], 0)]
    q = next((x for x in quality_rows if x["job_id"] == j["id"] and x["prompt_idx"] == 0), {})
    src = Image.open(j["img"]).convert("RGB").resize((512, 512))
    axes[r, 0].imshow(src)
    axes[r, 0].set_title(f"Input — {j['id']}", fontsize=9)
    axes[r, 0].axis("off")
    axes[r, 1].imshow(Image.open(pc[REFERENCE]))
    axes[r, 1].set_title(f"fp32 (GT)\n{j['edits'][0][:30]}", fontsize=9)
    axes[r, 1].axis("off")
    title = f"fp16+cache\nPSNR {q.get('psnr', float('nan')):.1f}  SSIM {q.get('ssim', float('nan')):.3f}"
    if "lpips" in q:
        title += f"  LPIPS {q['lpips']:.3f}"
    axes[r, 2].imshow(Image.open(pc[_imp]))
    axes[r, 2].set_title(title, fontsize=9)
    axes[r, 2].axis("off")
plt.tight_layout()
grid = OUT_EXP / "images" / "comparison_grid.png"
plt.savefig(grid, dpi=110, bbox_inches="tight")
plt.show()
print("Grid ->", grid)

## ⑧ Đóng gói kết quả (.zip) để tải về

Nén toàn bộ thư mục `experimental_data/quality_speed_bench_<ngày>/` (report + CSV + ảnh).
Trên **Colab** sẽ tự bật hộp thoại tải về; trên **Mac** in đường dẫn file zip.

In [ ]:
import shutil

# Nén OUT_EXP -> <OUT_EXP>.zip (gồm report.md, timing_raw.csv, quality_raw.csv, images/)
zip_base = OUT_EXP.parent / OUT_EXP.name  # bỏ phần mở rộng; make_archive tự thêm .zip
zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=OUT_EXP.parent, base_dir=OUT_EXP.name)
zip_path = Path(zip_path)
print(f"Đã nén: {zip_path}  ({zip_path.stat().st_size / 1e6:.2f} MB)")

if IN_COLAB:
    from google.colab import files

    files.download(str(zip_path))  # bật hộp thoại tải về trên trình duyệt
else:
    print("Mac: mở file zip ở đường dẫn trên (hoặc kéo từ trình quản lý file).")